In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor, VotingRegressor, StackingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("laptop_data.csv")
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [3]:
df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

In [4]:
df.head()

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [5]:
df.shape

(1303, 11)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1303 non-null   str    
 1   TypeName          1303 non-null   str    
 2   Inches            1303 non-null   float64
 3   ScreenResolution  1303 non-null   str    
 4   Cpu               1303 non-null   str    
 5   Ram               1303 non-null   str    
 6   Memory            1303 non-null   str    
 7   Gpu               1303 non-null   str    
 8   OpSys             1303 non-null   str    
 9   Weight            1303 non-null   str    
 10  Price             1303 non-null   float64
dtypes: float64(2), str(9)
memory usage: 250.8 KB


In [7]:
df.duplicated().sum()

np.int64(29)

In [8]:
df.nunique().duplicated().sum()

np.int64(1)

In [9]:
df["Ram"] = df["Ram"].str.replace("GB", "", regex=False).astype("int32")
df["Weight"] = df["Weight"].str.replace("kg", "", regex=False).astype("float32")
df["ScreenType"] = df["ScreenResolution"].str.replace(r"\s*\d+\s*x\s*\d+\s*$", "", regex=True).str.strip().replace("", "Unknown")
resolution = df["ScreenResolution"].str.extract(r"(\d+)\s*x\s*(\d+)")
df["Resolution"] = resolution[0] + "x" + resolution[1]
df["ppi"] = np.sqrt(resolution[0].astype(float)**2 + resolution[1].astype(float)**2) / df["Inches"]

In [10]:
memory = df["Memory"].fillna("")
for storage_type in ["HDD", "SSD"]:
    match = memory.str.extract(rf"(\d+(?:\.\d+)?)\s*(GB|TB)\s*{storage_type}", expand=False)
    df[storage_type] = (pd.to_numeric(match[0], errors="coerce").fillna(0) * match[1].map({"GB": 1, "TB": 1000}).fillna(1)).astype("int32")

In [11]:
df.drop(columns=["ScreenResolution", "Memory"], inplace=True)
df = df.dropna().copy()

In [12]:
df.head()

,Company,TypeName,Inches,Cpu,Ram,Gpu,OpSys,Weight,Price,ScreenType,Resolution,ppi,HDD,SSD
0,Apple,Ultrabook,13.3,Intel Core i5 2.3GHz,8,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832,IPS Panel Retina Display,2560x1600,226.983005,0,128
1,Apple,Ultrabook,13.3,Intel Core i5 1.8GHz,8,Intel HD Graphics 6000,macOS,1.34,47895.5232,Unknown,1440x900,127.677940,0,0
2,HP,Notebook,15.6,Intel Core i5 7200U 2.5GHz,8,Intel HD Graphics 620,No OS,1.86,30636.0000,Full HD,1920x1080,141.211998,0,256
3,Apple,Ultrabook,15.4,Intel Core i7 2.7GHz,16,AMD Radeon Pro 455,macOS,1.83,135195.3360,IPS Panel Retina Display,2880x1800,220.534624,0,512
4,Apple,Ultrabook,13.3,Intel Core i5 3.1GHz,8,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080,IPS Panel Retina Display,2560x1600,226.983005,0,256


In [13]:
X = df.drop(columns=["Price"])
y = np.log(df["Price"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=2)

In [14]:
X

,Company,TypeName,Inches,Cpu,Ram,Gpu,OpSys,Weight,ScreenType,Resolution,ppi,HDD,SSD
0,Apple,Ultrabook,13.3,Intel Core i5 2.3GHz,8,Intel Iris Plus Graphics 640,macOS,1.37,IPS Panel Retina Display,2560x1600,226.983005,0,128
1,Apple,Ultrabook,13.3,Intel Core i5 1.8GHz,8,Intel HD Graphics 6000,macOS,1.34,Unknown,1440x900,127.677940,0,0
2,HP,Notebook,15.6,Intel Core i5 7200U 2.5GHz,8,Intel HD Graphics 620,No OS,1.86,Full HD,1920x1080,141.211998,0,256
3,Apple,Ultrabook,15.4,Intel Core i7 2.7GHz,16,AMD Radeon Pro 455,macOS,1.83,IPS Panel Retina Display,2880x1800,220.534624,0,512
4,Apple,Ultrabook,13.3,Intel Core i5 3.1GHz,8,Intel Iris Plus Graphics 650,macOS,1.37,IPS Panel Retina Display,2560x1600,226.983005,0,256
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,Lenovo,2 in 1 Convertible,14.0,Intel Core i7 6500U 2.5GHz,4,Intel HD Graphics 520,Windows 10,1.80,IPS Panel Full HD / Touchscreen,1920x1080,157.350512,0,128
1299,Lenovo,2 in 1 Convertible,13.3,Intel Core i7 6500U 2.5GHz,16,Intel HD Graphics 520,Windows 10,1.30,IPS Panel Quad HD+ / Touchscreen,3200x1800,276.053530,0,512
1300,Lenovo,Notebook,14.0,Intel Celeron Dual Core N3050 1.6GHz,2,Intel HD Graphics,Windows 10,1.50,Unknown,1366x768,111.935204,0,0
1301,HP,Notebook,15.6,Intel Core i7 6500U 2.5GHz,6,AMD Radeon R5 M330,Windows 10,2.19,Unknown,1366x768,100.454670,1000,0


In [15]:
y

0       11.175755
1       10.776777
2       10.329931
3       11.814476
4       11.473101
          ...    
1298    10.433899
1299    11.288115
1300     9.409283
1301    10.614129
1302     9.886358
Name: Price, Length: 1303, dtype: float64

In [16]:
categorical_features = ["Company", "TypeName", "ScreenType", "Resolution", "Cpu", "Gpu", "OpSys"]
numerical_features = ["Inches", "Ram", "Weight", "ppi", "HDD", "SSD"]
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"), categorical_features),
    ("numerical", StandardScaler(), numerical_features)
])

In [17]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=10),
    "Lasso": Lasso(alpha=0.001),
    "KNN": KNeighborsRegressor(n_neighbors=3),
    "Decision Tree": DecisionTreeRegressor(max_depth=8),
    "SVR": SVR(kernel="rbf", C=10000, epsilon=0.1),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=3, max_samples=0.5, max_features=0.75, max_depth=15, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(n_estimators=100, random_state=3, bootstrap=True, max_samples=0.5, max_features=0.75, max_depth=15, n_jobs=-1),
    "AdaBoost": AdaBoostRegressor(n_estimators=15, learning_rate=1.0),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=500),
    "XGBoost": XGBRegressor(n_estimators=45, max_depth=5, learning_rate=0.5, n_jobs=-1)
}

In [18]:
rf = RandomForestRegressor(n_estimators=350, random_state=3, max_samples=0.5, max_features=0.75, max_depth=15, n_jobs=-1)
gbdt = GradientBoostingRegressor(n_estimators=100, max_features=0.5)
xgb = XGBRegressor(n_estimators=25, learning_rate=0.3, max_depth=5, n_jobs=-1)
et = ExtraTreesRegressor(n_estimators=100, random_state=3, bootstrap=True, max_samples=0.5, max_features=0.75, max_depth=10, n_jobs=-1)
models["Voting"] = VotingRegressor([("rf", rf), ("gbdt", gbdt), ("xgb", xgb), ("et", et)], weights=[5, 1, 1, 1], n_jobs=-1)
models["Stacking"] = StackingRegressor([("rf", rf), ("gbdt", gbdt), ("xgb", xgb)], final_estimator=Ridge(alpha=100), n_jobs=-1)

In [19]:
results = {}
trained_models = {}
for name, model in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results[name] = {"R2": r2_score(y_test, y_pred), "MAE": mean_absolute_error(np.exp(y_test), np.exp(y_pred))}
    trained_models[name] = pipeline

c:\Users\arnab\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\arnab\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\arnab\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\arnab\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 

In [20]:
results_df = pd.DataFrame(results).T.sort_values("R2", ascending=False)
print(results_df)

                         R2           MAE
Gradient Boosting  0.898675   8656.686496
XGBoost            0.898518   8705.697435
KNN                0.895475   8845.485482
Voting             0.890696   8950.752368
Random Forest      0.890516   8926.032022
Extra Trees        0.889532   9116.959040
Stacking           0.880495   9437.160798
Linear Regression  0.859139  10358.197124
Lasso              0.851653  10818.754118
Ridge              0.850094  11025.544788
SVR                0.840595  10836.787610
Decision Tree      0.815487  11230.038328
AdaBoost           0.745768  13620.491683


In [21]:
final_model = trained_models["Stacking"]
with open("df.pkl", "wb") as f:
    pickle.dump(df, f)
with open("pipe.pkl", "wb") as f:
    pickle.dump(final_model, f)